# Study 934 — Lump Sum vs DCA 💸

**A windfall lands. Send it all in on Monday, or drip it in over a year?**

The most repeated piece of retail money advice makes three promises at once: averaging in
gets you a *better average price*, *less risk*, and *more money*. We test all three on
the tape, over **every start month** of the sample.

The setup: $1, twelve months, valued on the same terminal date either way. **Lump sum**
buys the whole dollar at the start. **DCA** buys 1/12 at each of twelve month-ends, and —
unlike almost every published version of this test — the money still waiting sits in
**BIL** and earns the **real T-bill yield** (0% for six of these years, ~5% for three).
One execution lag: decided at a month-end close, filled at the next day's close.

Real tape: **SPY vs BIL**, daily total-return closes, 2007-05-30 → 2026-06-30
(217 start months), 1 bp one-way. Bond-heavy variant: **IEF vs BIL**.

*Every real number below is frozen from `docs/results.md` (SPY fingerprint
`edef65f148a6`); the live cells run only the offline synthetic control. As-of 2026-06-30.*


## 1. Who finishes richer?

Roll the whole experiment forward one month at a time and count. Over 217 start months from 2007-06-01 to 2026-06-01:

In [1]:
R = {'win': 76.0, 'win_lo': 69.9, 'win_hi': 81.2, 'mean_gap': 5.05, 'median_gap': 6.09, 'lump_mean': 12.35, 'dca_mean': 7.29, 't_hac': 3.19}
print(f"lump sum finishes richer in {R['win']:.1f}% of start months  "
      f"(95% CI {R['win_lo']:.1f}%-{R['win_hi']:.1f}%)")
print(f"average gap: {R['mean_gap']:+.2f} cents on every dollar invested   "
      f"(median {R['median_gap']:+.2f})")
print(f"average 12-month outcome: lump {R['lump_mean']:+.2f}%   vs   DCA {R['dca_mean']:+.2f}%")

lump sum finishes richer in 76.0% of start months  (95% CI 69.9%-81.2%)
average gap: +5.05 cents on every dollar invested   (median +6.09)
average 12-month outcome: lump +12.35%   vs   DCA +7.29%


## 2. Why — and it is not a market view

Nothing here is a forecast. Stocks pay a premium *on average*, and every dollar sitting in the queue is a dollar not being paid it. The T-bill yield the waiting money earns closes part of the gap but nowhere near all of it: crediting the real BIL path instead of the usual 0% assumption gives DCA back **0.59 cents** of a **5.64-cent** lead. 

> 🔬 **For the quants** — the windows overlap by up to eleven months, so the headline *t* is Newey-West with 12 lags (**+3.19**), backed by a fully non-overlapping check (**+2.18**) and a 12-month block bootstrap CI of **[+1.57, +7.90]** cents.

## 3. The part of the advice that is true

DCA really does lower risk. Its outcomes land in a band a little over **half** as wide as the lump sum's (dispersion ratio **0.587**), and its worst twelve months lose **-36.1%** where the lump sum's lose **-45.9%**. That is genuine — but look at *how* it is bought: by holding less equity for longer. The calm is a property of the **weight**, not of the schedule, so you can have exactly the same ride by picking a smaller stock weight and investing it at once — no twelve-month queue required. What you cannot do is keep the five cents *and* the calm: those cents are the extra risk. Section 3b puts a number on that.

In [2]:
R = {'sd_lump': 0.1701, 'sd_dca': 0.0999, 'disp_ratio': 0.587, 'lump_worst': -45.85, 'dca_worst': -36.13, 'worst_gap': -30.65, 'best_gap': 43.93}
print(f"spread of outcomes : lump {R['sd_lump']:.4f}   DCA {R['sd_dca']:.4f}   "
      f"-> ratio {R['disp_ratio']:.3f}")
print(f"worst 12 months    : lump {R['lump_worst']:.1f}%   DCA {R['dca_worst']:.1f}%")
print(f"worst / best gap   : {R['worst_gap']:+.1f} cents / {R['best_gap']:+.1f} cents")
print( '                     -> going all-in is right three times in four,'
       ' and expensively wrong the fourth')

spread of outcomes : lump 0.1701   DCA 0.0999   -> ratio 0.587
worst 12 months    : lump -45.9%   DCA -36.1%
worst / best gap   : -30.6 cents / +43.9 cents
                     -> going all-in is right three times in four, and expensively wrong the fourth


## 3b. So what are those five cents, really?

Not a timing skill — an ownership difference. Spread over twelve months, the DCA plan owns the market for only **54.2%** of the year on average (the first tranche is invested all twelve months, the last one for none of it). So put that number on the table and race DCA against the *boring* portfolio: **54.2% in stocks and the rest in T-bills, bought at the start and left alone**.

The five cents vanish: **-0.04 cents**, a coin flip (53.5% of months), with an interval of [-1.18, +0.98] cents around zero. DCA was never buying worse prices — it was just owning less stock, and it was paid accordingly.

In [3]:
R = {'mean_gap': 5.05, 'em_w': 54.2, 'em_gap': -0.04, 'em_win': 53.5, 'em_lo': -1.18, 'em_hi': 0.98, 'sd_dca': 0.0999, 'em_sd': 0.0926}
print(f"lump sum (100% invested) vs DCA        : {R['mean_gap']:+.2f} cents")
print(f"static {R['em_w']:.1f}% stocks + bills vs DCA : {R['em_gap']:+.2f} cents  "
      f"({R['em_win']:.1f}% of months, CI [{R['em_lo']:+.2f}, {R['em_hi']:+.2f}])")
print(f"same calm ride                         : spread {R['em_sd']:.4f} vs DCA {R['sd_dca']:.4f}")
print()
print('-> the twelve tranches add nothing the weight had not already given you')

lump sum (100% invested) vs DCA        : +5.05 cents
static 54.2% stocks + bills vs DCA : -0.04 cents  (53.5% of months, CI [-1.18, +0.98])
same calm ride                         : spread 0.0926 vs DCA 0.0999

-> the twelve tranches add nothing the weight had not already given you


## 4. "But surely DCA wins when the market looks expensive?"

That is the version of the advice worth testing, so we tested it — with hindsight, which is the friendliest possible framing. Split the start months by how stretched SPY was against its own three-year average (a **price proxy**, not CAPE), and separately by whether you were starting inside a drawdown:

| Starting from | lump wins | average gap |
|---|--:|--:|
| a cheap market | 93.4% | +10.10c |
| a middling market | 80.0% | +5.64c |
| a **stretched** market | 73.3% | +3.43c |
| **10%+ below the high** | 72.9% | +5.26c |

The advantage shrinks where the story says it should — and **never crosses zero**. There was no starting condition, in nineteen years, in which drip-feeding beat sending it. Averaging in is not buying the fear; it is just owning less.

## 5. Two more things worth knowing

**Taking longer costs more.** Three tranches costs +0.81c, six +2.18c, twelve +5.05c, twenty-four **+11.61c** — and each of them smooths the ride a little more. The comfort and the bill scale together; pick your point on that line knowingly.

**Bonds are a different question.** Put the windfall into a Treasury sleeve (IEF) instead and the lump sum's edge collapses to +1.02c, no longer distinguishable from zero. The prize was never a trick of timing — it was the risk premium of whatever you were buying, collected sooner. Small premium, small prize.

## 6. Live check — the machinery has no thumb on the scale (offline synthetic)

Before believing a result that agrees this neatly with theory, make the harness prove it can say the opposite. On simulated tapes with a **planted** premium the lump sum must win; with **no** premium the answer must sit on a coin flip; on a **falling** tape DCA must win. Twelve independent 25-year paths per world.

In [4]:
import os, sys
sys.path.insert(0, os.path.abspath('..'))
sys.path.insert(0, os.path.abspath(os.path.join('..','..','..')))
from lump_vs_dca import data, strategy as st

def small(signal_strength, seed):
    return data.synthetic_daily(n_years=15, signal_strength=signal_strength, seed=seed)

for ss, label in [(1.0, 'rising  (premium planted)'),
                  (0.0, 'flat    (the null)     '),
                  (-1.0, 'falling (premium is negative)')]:
    c = st.synthetic_control(ss, seeds=range(934, 940), synth=small)
    print('%s : mean gap %+6.2f cents, lump wins on %3.0f%% of paths'
          % (label, c['mean_gap_cents'], 100 * c['frac_seeds_lump_wins']))

rising  (premium planted) : mean gap  +4.45 cents, lump wins on 100% of paths


flat    (the null)      : mean gap  +0.76 cents, lump wins on  67% of paths


falling (premium is negative) : mean gap  -2.58 cents, lump wins on  17% of paths


## Verdict

- **Signal — Real.** The lump sum wins **76.0%** of start months by **+5.05 cents on the dollar**, HAC *t* = **+3.19**, bootstrap CI [+1.57, +7.90] clear of zero, same sign in every cut and on the 2000-2026 history (*t* = +3.25). The advice is not merely unproven; it is backwards.
- **Tradability — Mirage.** There is nothing to bank. Match the exposure — a static 54.2% stock portfolio against DCA — and the whole gap is -0.04 cents (*t* = -0.08, interval [-1.18, +0.98]). It is beta you were always paid for, which is why it dies on bonds (+1.02c) and died through the 2000s (+0.67c, *t* = +0.26).
- **Does DCA lower risk? — Confirmed.** Dispersion ratio 0.587, worst window -36.1% vs -45.9%. Real, and available more cheaply by owning less stock.